# Prefect + Dask

This is a tutorial to use the Prefect and Dask clusters together from Jupyter.

NOTE: we can use the EOPF Dask cluster from Prefect, but not the staging cluster because of dependency conflicts.

See the associated Python module: [my_prefect_and_dask.py](./my_prefect_and_dask.py)

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

# Init cluster from this venv kernel
from resources.dask_clusters.dask_venv import init_dask_cluster_mockup_venv
dask_gateway, dask_cluster, dask_client = init_dask_cluster_mockup_venv(
    scale=1,
    worker_cores=1,
    worker_memory=2.0,
    scheduler_memory_limit=2,
)

In [ ]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3"

In [ ]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import resources

# Data to test the example flow
my_data = {"start": "2000", "end": "2001", "freq": "2w"}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

In [ ]:
# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

# 1. Call Prefect flow from Python code

In [ ]:
# Import the module, or reload it if you changed its source code
import my_prefect_and_dask
reload(my_prefect_and_dask)

# Call flow and print results
result = my_prefect_and_dask.my_flow(**my_data)
#display(result.head()) # we have strange errors with this line, I don't know why, maybe a dependency version problem

In [ ]:
# Visualize the dask graph
result.visualize(filename=None, tasks=False)

<div class="alert alert-info" role="alert">
Notes:

  1. The dask logs are not always printed, sometimes yes sometimes no, I couldn't find why...
  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.
  1. Check in the dashboard and above logs that when we call the flow from Python code, no Prefect workers are involved:
      1. Your module main code (outside functions) and flow are run only by client.
      1. The tasks are run only by the Dask workers.
  1. If you want to check the IP adresses:
      1. On kubernetes, you can run the `kubectl describe` command to check a pod IP address.
      1. In local mode, use: `docker inspect <container_id> | grep IPAddress`

# 2. Deploy Prefect flow

See the full yaml file: [deploy-prefect-dask.yaml](./deploy-prefect-dask.yaml)

First we deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

# Use a specific secret block on the bucket for this subfolder
code_bucket, os.environ["SHARE_BUCKET"] = await get_share_bucket(s3_code_folder)

if local_mode:
    print (f"S3 SeaweedFS dashboard: http://localhost:9101 with user=seaweedfs password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{code_bucket.bucket_name}/{code_bucket.bucket_folder}'")

# Upload local directory and resources contents
await code_bucket.put_directory(local_path = ".", to_path = ".")
await code_bucket.put_directory(local_path = resources.__path__[0], to_path = "resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = code_bucket.bucket_folder

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./deploy-prefect-dask.yaml"

In [ ]:
deploy_name = "my-flow/tuto-prefect-dask"
await prefect_utils.wait_for_deployment(deploy_name)

In [ ]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.
  1. Check in the dashboard and above logs that:
      1. Your module main code (outside functions) is run by the client and Prefect workers, not Dask workers.
      2. The flow is run only by the Prefect workers.
      1. The tasks are run only by the Dask workers.

## 3. Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
from resources.dask_clusters.dask_utils import *
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)
    close_dask_clusters(dask_gateway, dask_cluster, client)